In [ ]:
##modules
#%matplotlib widget
#%matplotlib inline

%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('Qt5Agg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

In [ ]:
# Variables path
layer_script = "block"

#process=""
disco="g"

subj = "sub-A2004"
subj_copy=subj+"_copia"

#subj = sys.argv[1] ## name of participant list

# Carpeta general
datadir = Path(f"{disco}:\MOUS_204")

#carpetas generales de datos
mri_dir = datadir / f"{subj}"/"anat"
meg_dir = datadir / f"{subj}"/"meg"

# Carpeta de preprocesado
output_preproc = datadir / "output_preproc"

preproc_path = output_preproc / f"preproc_{layer_script}"
preproc_path.mkdir(parents=True, exist_ok=True)
 
# Carpeta de epocas "sucias"
epochs_path = preproc_path / f"epochs_{layer_script}"
epochs_path.mkdir(parents=True, exist_ok=True) 

# Carpeta de ICA
ICA_path = preproc_path / f"ICA_{layer_script}"
ICA_path.mkdir(parents=True, exist_ok=True)

# Épocas limpias
epochs_clean_path = preproc_path / f"epochs_clean_{layer_script}"
epochs_clean_path.mkdir(parents=True, exist_ok=True)

#epocas evoked
evoked_path = Path(preproc_path) / f"evoked_{layer_script}"
evoked_path.mkdir(parents=True, exist_ok=True)
 

# Definir la carpeta de output_source antes de usarla
output_source = Path(r"g:\MOUS_204\output_source")

source_path = output_source / f"source_{layer_script}"
source_path.mkdir(parents=True, exist_ok=True)

#raw_hsp es el raw con fiducials cargados
raw_hsp_path = source_path / f"raw_hsp"
raw_hsp_path.mkdir(parents=True, exist_ok=True)

# Carpeta de forward solution
fwd_path = source_path / f"fwd"
fwd_path.mkdir(parents=True, exist_ok=True)

# Carpeta de inverse solution
inverse_path = source_path / f"inverse"
inverse_path.mkdir(parents=True, exist_ok=True)


mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)


# epochs_zinnen_path = epochs_clean_path / f"{subj}_epochs_zinnen_{type_script}"
# epochs_woorden_path = epochs_clean_path / f"{subj}_epochs_woorden_{type_script}"

data_zinnen= epochs_clean_path / f"{subj}_epochs_zinnen_{type_script}-epo.fif"

epochs_zinnen = mne.read_epochs(data_zinnen, preload=True) 




Reading g:\MOUS_204\output_preproc\block_preproc_0.5_40\sub-A2002_preproc_block\sub-A2002_epochs_clean_block\sub-A2002_epochs_zinnen_block\sub-A2002_epochs_clean_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   40000.00 ms
        0 CTF compensation matrices available
Not setting metadata
22 matching events found
No baseline correction applied
0 projection items activated


Number of events,22
Events,40second: 22
Time range,0.000 – 40.000 s
Baseline,off


In [11]:
len(epoch[:,1])

273

In [19]:
## prueba fuera de la funcion

from statsmodels.tsa.stattools import acf

#transform epochs into array
epochs_data= epochs_zinnen.get_data()

duration= epochs_zinnen.tmax - epochs_zinnen.tmin
sfreq= epochs_zinnen.info['sfreq']

lags=duration*sfreq
#get one epoch called epoch
acw_0_channels = {}
acw_50_channels = {}
for j in range(0,len(epochs_data)):
    epoch= epochs_data[0]
    acw_0_channels[f'acw_0_channel_{j}'] = []
    acw_50_channels[f'acw_50_channel_{j}'] = []

    for i in range(0,len(epoch[:,1])):
        #get one channel, as ACF is going to be calculated for each channel

        channel=epoch[i]
    #time series is going to be the x in acf
        x=channel
            # Calcular la función de autocorrelación
                # statsmodels.tsa.stattools.acf(x, adjusted=False, nlags=None, qstat=False,/
                # fft=True, alpha=None, bartlett_confint=True, missing='none'  n
                #bartlett_confint i dont know, qstats is not necessary

        acf_value=acf(x, adjusted=True,fft=True, alpha=0.05, missing="conservative")


        ACW_50_i = np.argmax(acf_value[0] <= 0.5)
        acw_50 = ACW_50_i / sfreq

        acw_50_channels[f'acw_50_channel_{j}'].append(acw_50)


        ACW_0_i = np.argmax(acf_value[0] <= 0)
        acw_0 = ACW_0_i / sfreq
        acw_0_channels[f'acw_0_channel_{j}'].append(acw_0)




C:\Users\UCM\AppData\Local\Temp\ipykernel_4464\1836704230.py:6: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  epochs_data= epochs_zinnen.get_data()


In [20]:

# Calcular la media de ACW-0 para todos los canales
acw_0_all_values = [np.mean(acw_0_channels[channel]) for channel in acw_0_channels]
acw_0_mean = np.mean(acw_0_all_values)

# Calcular la media de ACW-50 para todos los canales
acw_50_all_values = [np.mean(acw_50_channels[channel]) for channel in acw_50_channels]
acw_50_mean = np.mean(acw_50_all_values)

# Imprimir los resultados
print(f"Media de ACW-0 en todos los canales: {acw_0_mean}")
print(f"Media de ACW-50 en todos los canales: {acw_50_mean}")

Media de ACW-0 en todos los canales: 0.008296703296703295
Media de ACW-50 en todos los canales: 0.017216117216117214


In [28]:
# Crear una figura con 4 subplots (2x2)
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

# Subplot 1: ACF value lags
axs[0, 0].plot(acf_value_lags[0], label="lags")
axs[0, 0].set_title("ACF Value Lags")
axs[0, 0].set_xlabel("Lag")
axs[0, 0].set_ylabel("Autocorrelation")

# Subplot 2: ACF value default
axs[0, 1].plot(acf_value_default[0], label="default", color='orange')
axs[0, 1].set_title("ACF Value Default")
axs[0, 1].set_xlabel("Lag")
axs[0, 1].set_ylabel("Autocorrelation")

# Subplot 3: ACF value 1000
axs[1, 0].plot(acf_value_1000[0], label="1000", color='green')
axs[1, 0].set_title("ACF Value 1000")
axs[1, 0].set_xlabel("Lag")
axs[1, 0].set_ylabel("Autocorrelation")

# Subplot 4: ACF value 10000
axs[1, 1].plot(acf_value_10000[0], label="10000", color='red')
axs[1, 1].set_title("ACF Value 10000")
axs[1, 1].set_xlabel("Lag")
axs[1, 1].set_ylabel("Autocorrelation")

# Ajustar el layout para evitar que se solapen los subplots
plt.tight_layout()

# Mostrar la figura
plt.show()

In [ ]:
#ACW reviewed definitiva

import statsmodels.tsa.stattools.acf as acf

def acw(x, fs, missing="conservative", isplot=False):
    """
    Calcula la ventana de autocorrelación (ACW) de una serie temporal.
    
    Parámetros:
    x : array_like
        Serie temporal. , es posible que lo ajuste para que se calcule para cada electrodo
        
    fs : float
        Frecuencia de muestreo.
    isplot : bool, opcional
        Si True, grafica la función de autocorrelación. El valor por defecto es False.
    
    Retorna:
    acw_0 : float
        ACW-0.
    acw_50 : float
        ACW-50.
    acf : ndarray
        Función de autocorrelación (eje y).
    lags : ndarray
        Lags (eje x).
    """
    
    # Calcular la función de autocorrelación
        # statsmodels.tsa.stattools.acf(x, adjusted=False, nlags=None, qstat=False,/
        # fft=True, alpha=None, bartlett_confint=True, missing='none'


    for i in range(len(x)):
        lags = np.arange(len(acf)) / fs
        acf= acf(x, nlags= lags)
        # acf = correlate(x, x, mode='full', method='auto')
        # acf = acf[len(acf)//2:]  # Considerar solo la segunda mitad (lags positivos)
        # acf /= np.max(acf)  # Normalizar la autocorrelación



        # Encontrar ACW-50 y ACW-0
        ACW_50_i = np.argmax(acf <= 0.5)
        acw_50 = ACW_50_i / fs
        
        ACW_0_i = np.argmax(acf <= 0)
        acw_0 = ACW_0_i / fs

        # Graficar si se solicita
        if isplot:
            plt.figure()
            plt.plot(lags, acf, 'k')
            plt.xlim([0, np.max(lags)])
            plt.fill_between(lags[:ACW_50_i], acf[:ACW_50_i], color='r', alpha=0.3)
            plt.fill_between(lags[:ACW_0_i], acf[:ACW_0_i], color='m', alpha=0.3)
            plt.title(f'ACW-0 = {acw_0:.1f} s, ACW-50 = {acw_50:.1f} s')
            plt.xlabel('Lags (s)')
            plt.ylabel('Autocorrelation')
            plt.show()

        return acw_0, acw_50, acf, lags



In [ ]:
def calculate_acf_for_epochs(epochs, isplot=False, print_results=False):
    fs = epochs.info['sfreq']  # Obtener la frecuencia de muestreo una vez
    epoch_data = epochs.get_data()  # Obtener los datos de todas las épocas

    
    all_acw_0, all_acw_50, all_acf, all_lags = [], [], [], []

    for idx, e in enumerate(epoch_data):
        series_temporal = e[0, :]  # Primera canal, todos los puntos temporales
        acw_0, acw_50, acf, lags = acf(series_temporal, fs, isplot)
        all_acw_0.append(acw_0)
        all_acw_50.append(acw_50)
        all_acf.append(acf)
        all_lags.append(lags)
        
        if print_results:
            print(f"Epoch {idx}")
            print(f"ACW-0: {acw_0}")
            print(f"ACW-50: {acw_50}")
            print(f"ACF: {acf}")
            print(f"Lags: {lags}")
            print()  # Imprimir una línea en blanco para separar las épocas

    return all_acw_0, all_acw_50, all_acf, all_lags


In [93]:
acw_0, acw_50, acf, lags= calculate_acf_for_epochs(epochs, isplot=False, print_results=True)




Epoch 0
ACW-0: 0.4683333333333333
ACW-50: 0.013333333333333334
ACF: [ 1.00000000e+00  9.16062399e-01  8.17187881e-01 ... -1.19933907e-04
 -7.64772707e-05 -4.02200808e-05]
Lags: [0.00000000e+00 1.66666667e-03 3.33333333e-03 ... 3.99966667e+01
 3.99983333e+01 4.00000000e+01]

Epoch 1
ACW-0: 1.6316666666666666
ACW-50: 0.013333333333333334
ACF: [ 1.00000000e+00  9.33576488e-01  8.51696748e-01 ... -1.36197685e-04
 -9.58217358e-05 -5.81087157e-05]
Lags: [0.00000000e+00 1.66666667e-03 3.33333333e-03 ... 3.99966667e+01
 3.99983333e+01 4.00000000e+01]

Epoch 2
ACW-0: 0.0
ACW-50: 0.0
ACF: [nan nan nan ... nan nan nan]
Lags: [0.00000000e+00 1.66666667e-03 3.33333333e-03 ... 3.99966667e+01
 3.99983333e+01 4.00000000e+01]

Epoch 3
ACW-0: 0.0
ACW-50: 0.0
ACF: [nan nan nan ... nan nan nan]
Lags: [0.00000000e+00 1.66666667e-03 3.33333333e-03 ... 3.99966667e+01
 3.99983333e+01 4.00000000e+01]

Epoch 4
ACW-0: 1.1266666666666667
ACW-50: 0.013333333333333334
ACF: [ 1.00000000e+00  9.26462855e-01  8.371070

C:\Users\UCM\AppData\Local\Temp\ipykernel_6960\1005335462.py:3: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  epoch_data = epochs.get_data()  # Obtener los datos de todas las épocas
c:\Users\UCM\anaconda3\Lib\site-packages\scipy\signal\_signaltools.py:242: RuntimeWarning: Use of fft convolution on input with NAN or inf results in NAN or inf output. Consider using method='direct' instead.
  return convolve(in1, _reverse_and_conj(in2), mode, method)


In [97]:
for i in range(0, len(epochs)):
    print(acw_50[i])


0.013333333333333334
0.013333333333333334
0.0
0.0
0.013333333333333334
0.015
0.016666666666666666
0.013333333333333334
0.013333333333333334
0.0
0.013333333333333334
0.013333333333333334
0.013333333333333334
0.013333333333333334
0.011666666666666667
0.013333333333333334
0.013333333333333334
0.013333333333333334
0.013333333333333334
0.013333333333333334
0.013333333333333334
0.015
0.0
0.011666666666666667


In [98]:
for i in range(0, len(epochs)):
    print(acw_0[i])

0.4683333333333333
1.6316666666666666
0.0
0.0
1.1266666666666667
1.5583333333333333
1.6633333333333333
0.9366666666666666
1.2833333333333334
0.0
1.4866666666666666
0.6133333333333333
0.6016666666666667
0.4583333333333333
0.34
0.61
0.8433333333333334
0.8833333333333333
2.28
1.3433333333333333
0.445
2.035
0.0
0.4483333333333333


In [ ]:
#ACF DEFINITION yashir

def acw(x, fs, isplot=False):
    """
    Calcula la ventana de autocorrelación (ACW) de una serie temporal.
    
    Parámetros:
    x : array_like
        Serie temporal.
    fs : float
        Frecuencia de muestreo.
    isplot : bool, opcional
        Si True, grafica la función de autocorrelación. El valor por defecto es False.
    
    Retorna:
    acw_0 : float
        ACW-0.
    acw_50 : float
        ACW-50.
    acf : ndarray
        Función de autocorrelación (eje y).
    lags : ndarray
        Lags (eje x).
    """
    
    # Calcular la función de autocorrelación
    acf = correlate(x, x, mode='full', method='auto')
    acf = acf[len(acf)//2:]  # Considerar solo la segunda mitad (lags positivos)
    acf /= np.max(acf)  # Normalizar la autocorrelación

    # Crear el vector de lags
    lags = np.arange(len(acf)) / fs

    # Encontrar ACW-50 y ACW-0
    ACW_50_i = np.argmax(acf <= 0.5)
    acw_50 = ACW_50_i / fs
    
    ACW_0_i = np.argmax(acf <= 0)
    acw_0 = ACW_0_i / fs

    # Graficar si se solicita
    if isplot:
        plt.figure()
        plt.plot(lags, acf, 'k')
        plt.xlim([0, np.max(lags)])
        plt.fill_between(lags[:ACW_50_i], acf[:ACW_50_i], color='r', alpha=0.3)
        plt.fill_between(lags[:ACW_0_i], acf[:ACW_0_i], color='m', alpha=0.3)
        plt.title(f'ACW-0 = {acw_0:.1f} s, ACW-50 = {acw_50:.1f} s')
        plt.xlabel('Lags (s)')
        plt.ylabel('Autocorrelation')
        plt.show()

    return acw_0, acw_50, acf, lags